> 📘 **FES** · Fonaments d'Enginyeria de Software amb Python

# 12 · Disseny i composició

Una sessió d'una hora. El sistema de monitoratge conté diversos sensors simulats i coordina les lectures.

## Objectius d'aprenentatge

- Assignar responsabilitats a les classes.
- Relacionar objectes mitjançant composició.
- Interpretar classes, associacions i multiplicitats en UML.
- Generar un resum i tractar errors d'identificació.

Executa les cel·les en ordre des d'un kernel net.

## Responsabilitat d'un sensor

Recuperem la classe del bloc 11. Cada sensor conserva el seu estat i valida les lectures abans de registrar-les. El notebook inclou aquesta definició per funcionar de manera independent. En un projecte més gran la compartiríem en un mòdul.

In [ ]:
from math import isfinite


class Sensor:
    """Conserva i valida l'històric d'un sol sensor."""

    def __init__(self, nom, unitat):
        """Construeix un sensor sense lectures."""
        self.nom = nom
        self.unitat = unitat
        self.lectures = []

    def registrar(self, valor):
        """Afegeix un nombre finit o llança ValueError."""
        if isinstance(valor, bool) or not isinstance(valor, (int, float)):
            raise ValueError("La lectura ha de ser un nombre.")
        if isinstance(valor, float) and not isfinite(valor):
            raise ValueError("La lectura ha de ser finita.")
        self.lectures.append(valor)

    def ultima_lectura(self):
        """Retorna l'última lectura o None si encara no n'hi ha."""
        if not self.lectures:
            return None
        return self.lectures[-1]

    def mitjana(self):
        """Retorna la mitjana de l'històric o None."""
        if not self.lectures:
            return None
        return sum(self.lectures) / len(self.lectures)

## Un sistema conté sensors

La composició d'objectes consisteix a construir funcionalitat combinant objectes. `SistemaMonitoratge` conserva un diccionari de sensors. El nom és la clau i l'objecte és el valor.

`afegir_sensor` comprova la unicitat del nom. `registrar` busca el sensor i delega el registre al seu mètode. `resum` consulta l'última lectura i retorna dades senzilles que més endavant podrem guardar en JSON. El sistema s'encarrega de coordinar, i cada sensor gestiona el seu històric.

In [ ]:
class SistemaMonitoratge:
    """Organitza sensors per nom i coordina el registre de lectures."""

    def __init__(self):
        """Crea un sistema sense sensors."""
        self.sensors = {}

    def afegir_sensor(self, sensor):
        """Afegeix un sensor i rebutja un nom que ja existeixi."""
        if sensor.nom in self.sensors:
            raise ValueError("Ja existeix un sensor amb aquest nom.")
        self.sensors[sensor.nom] = sensor

    def registrar(self, nom, valor):
        """Envia una lectura al sensor indicat pel seu nom."""
        if nom not in self.sensors:
            raise ValueError("El sensor no existeix.")
        self.sensors[nom].registrar(valor)

    def resum(self):
        """Retorna una fila per sensor, en ordre d'incorporació."""
        files = []
        for sensor in self.sensors.values():
            files.append({
                "sensor": sensor.nom,
                "unitat": sensor.unitat,
                "valor": sensor.ultima_lectura(),
            })
        return files

## UML introductori

Les caixes indiquen el nom, els atributs i els mètodes de cada classe. La línia és una associació. `0..*` significa zero o més sensors per sistema.

```text
+-------------------------+       +----------------------+
| SistemaMonitoratge      |       | Sensor               |
+-------------------------+       +----------------------+
| sensors                 |       | nom                  |
|                         |-------| unitat               |
|                         |       | lectures             |
+-------------------------+       +----------------------+
| afegir_sensor(sensor)   |       | registrar(valor)     |
| registrar(nom, valor)   |       | ultima_lectura()     |
| resum()                 |       | mitjana()            |
+-------------------------+       +----------------------+

SistemaMonitoratge -------- 0..* Sensor
```

Fem servir composició d'objectes en codi. El diagrama utilitza una associació simple perquè els sensors es creen fora del sistema i poden existir independentment. Un diamant ple de composició UML afegiria propietat exclusiva i dependència del cicle de vida, que aquest exemple no imposa.

## Construir i consultar el sistema

El resum segueix l'ordre d'incorporació. H1 apareix amb `None` perquè encara no té lectures. T1 mostra la darrera de les dues.

In [ ]:
sistema = SistemaMonitoratge()
sistema.afegir_sensor(Sensor("T1", "°C"))
sistema.afegir_sensor(Sensor("H1", "%"))
sistema.registrar("T1", 22.5)
sistema.registrar("T1", 23.0)
print(sistema.resum())

## Errors i responsabilitats

Un nom duplicat o desconegut és responsabilitat del sistema. Una lectura invàlida és responsabilitat del sensor. Localitza els tres `raise` que s'activen en els intents següents. La validació es fa abans del canvi d'estat.

In [ ]:
try:
    sistema.afegir_sensor(Sensor("T1", "°C"))
except ValueError as error:
    print(error)

for nom, valor in [("V1", 3.3), ("T1", "error")]:
    try:
        sistema.registrar(nom, valor)
    except ValueError as error:
        print(error)
print(sistema.resum())

## Cas límit i discussió de disseny

Un sistema buit és un estat vàlid i produeix una llista buida. La responsabilitat única facilita localitzar els canvis. Explica quina classe modificaríem per afegir una validació numèrica i quina modificaríem per canviar el format del resum. Conservar les responsabilitats també afavoreix les proves del bloc 13.

In [ ]:
buit = SistemaMonitoratge()
print(buit.resum())  # []

## Pràctica

Els exercicis 1–3 construeixen el sistema, deleguen lectures i generen el resum. L'exercici 4 proposa alarmes per unitat com a repte opcional.

[Enunciats](01_sistema_monitoratge_exercicis.py) · [Solucions](01_sistema_monitoratge_solucions.py)

Les relacions i la composició són imprescindibles. L'herència s'estudia al bloc opcional 15.